In [1]:
import os
os.environ["LOKY_MAX_CPU_COUNT"] = "4"  # Adjust to your CPU's physical core count
import numpy as np
import pandas as pd
import mne
import matplotlib.pyplot as plt
from scipy import signal
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, confusion_matrix
import seaborn as sns
import glob
import time
import logging
import re
import tensorflow as tf
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv1D, MaxPooling1D, Flatten, Dense, Dropout, BatchNormalization

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# Set MNE verbosity
mne.set_log_level('ERROR')

# Define dataset parameters
data_dir = r'C:\Users\arnna\Desktop\fypcnn2\STEW Dataset'
fs = 128  # Sampling frequency in Hz
T = 150   # Duration in seconds
n_samples = int(T * fs)  # 19200 samples
ch_names = ['AF3', 'F7', 'F3', 'FC5', 'T7', 'P7', 'O1', 'O2', 'P8', 'T8', 'FC6', 'F4', 'F8', 'AF4']
n_channels = len(ch_names)

# Data augmentation parameters
window_size = 4 * fs  # 4 seconds = 512 samples
step_size = 2 * fs    # 2 seconds = 256 samples (50% overlap)

# Step 1: Data Labeling (Ternary Classification)
def create_labels_file(data_dir):
    ratings_file = os.path.join(data_dir, 'ratings.txt')
    class_dict = {}
    with open(ratings_file, 'r') as f:
        for line in f:
            parts = [part.strip() for part in line.split(',')]
            if len(parts) == 3:
                try:
                    subject_id, rating_lo, rating_hi = map(int, parts)
                    subject_id_str = str(subject_id).zfill(2)
                    class_lo = 0 if 1 <= rating_lo <= 3 else (1 if 4 <= rating_lo <= 6 else 2)
                    class_hi = 0 if 1 <= rating_hi <= 3 else (1 if 4 <= rating_hi <= 6 else 2)
                    class_dict[(subject_id_str, 'lo')] = class_lo
                    class_dict[(subject_id_str, 'hi')] = class_hi
                    logger.info(f"Subject {subject_id_str}: class_lo = {class_lo}, class_hi = {class_hi}")
                except ValueError:
                    logger.warning(f"Could not convert line to integers: {line.strip()}")

    data_files = [f for f in os.listdir(data_dir) if f.endswith('.txt') and f != 'ratings.txt']
    mappings = []
    for file_name in data_files:
        if file_name.startswith('sub') and file_name.endswith('.txt'):
            subject_id_str = file_name[3:5]
            task = file_name[6:8]
            if (subject_id_str, task) in class_dict:
                class_label = class_dict[(subject_id_str, task)]
                mappings.append((file_name, class_label))
            else:
                logger.warning(f"No class label found for {file_name}")

    df = pd.DataFrame(mappings, columns=['file_name', 'class_label'])
    output_file = os.path.join(data_dir, 'labels.csv')
    df.to_csv(output_file, index=False)
    logger.info(f"Created labels.csv with {len(df)} entries.")
    return df

# Load EEG data with labels
def load_eeg_data(data_dir, labels_df):
    data = []
    labels = []
    subject_ids = []
    conditions = []
    delimiters = [r'\s+', '\t', ',', ';']
    for _, row in labels_df.iterrows():
        file_name = row['file_name']
        label = row['class_label']
        file_path = os.path.join(data_dir, file_name)
        file_size = os.path.getsize(file_path) / 1024  # Size in KB
        logger.info(f"Processing {file_name} (Size: {file_size:.2f} KB)")
        if file_size < 10:  # Skip very small files
            logger.warning(f"Skipping {file_name}: File too small ({file_size:.2f} KB)")
            continue
        for sep in delimiters:
            try:
                eeg_data = pd.read_csv(file_path, sep=sep, header=None, engine='python')
                logger.info(f"Parsed {file_name} with delimiter: {sep}")
                if eeg_data.shape[0] < int(0.9 * n_samples) or eeg_data.shape[1] != n_channels:
                    logger.warning(f"Skipping {file_name}: Incorrect shape {eeg_data.shape}, expected (~{n_samples}, {n_channels})")
                    break
                if not np.all(eeg_data.apply(lambda x: np.issubdtype(x.dtype, np.number))):
                    logger.warning(f"Skipping {file_name}: Contains non-numeric data")
                    break
                eeg_data = eeg_data.iloc[:n_samples].values.T  # Transpose to channels x samples
                if np.any(np.isnan(eeg_data)):
                    logger.warning(f"Skipping {file_name}: Contains NaN values")
                    continue
                data.append(eeg_data)
                labels.append(label)
                subject_id = file_name[3:5]
                condition = 'high' if 'hi' in file_name.lower() else 'low'
                subject_ids.append(subject_id)
                conditions.append(condition)
                logger.info(f"Loaded {file_name}: Shape {eeg_data.shape}, Label: {label}")
                break
            except Exception as e:
                logger.error(f"Error reading {file_name} with delimiter {sep}: {str(e)}")
                if sep == delimiters[-1]:
                    logger.warning(f"Skipping {file_name}: Failed to parse with all delimiters")
                    try:
                        eeg_data = pd.read_fwf(file_path, header=None)
                        if eeg_data.shape[0] < int(0.9 * n_samples) or eeg_data.shape[1] != n_channels:
                            logger.warning(f"Skipping {file_name}: Incorrect shape {eeg_data.shape} with fixed-width")
                            break
                        eeg_data = eeg_data.iloc[:n_samples].values.T
                        if np.any(np.isnan(eeg_data)):
                            logger.warning(f"Skipping {file_name}: Contains NaN values with fixed-width")
                            continue
                        data.append(eeg_data)
                        labels.append(label)
                        subject_id = file_name[3:5]
                        condition = 'high' if 'hi' in file_name.lower() else 'low'
                        subject_ids.append(subject_id)
                        conditions.append(condition)
                        logger.info(f"Loaded {file_name}: Shape {eeg_data.shape} with fixed-width")
                    except Exception as e:
                        logger.error(f"Error reading {file_name} with fixed-width: {str(e)}")
    if not data:
        logger.error("No valid EEG files were loaded.")
        raise ValueError("No valid EEG files were loaded.")
    return np.array(data), np.array(labels), subject_ids, conditions

# Step 2: Preprocessing
def bandpass_filter(data, lowcut=1.0, highcut=40.0, fs=fs, order=5):
    nyq = 0.5 * fs
    low = lowcut / nyq
    high = highcut / nyq
    b, a = signal.butter(order, [low, high], btype='bandpass')
    filtered = signal.filtfilt(b, a, data, axis=-1)
    return filtered

def plot_eeg_channels(data, ch_names, fs, filename='eeg_channels_ml.png'):
    plt.figure(figsize=(15, 10))
    times = np.linspace(0, T, data.shape[1])
    for i, ch in enumerate(ch_names):
        plt.subplot(len(ch_names), 1, i+1)
        plt.plot(times, data[i], label=ch)
        plt.ylabel(ch)
        if i == len(ch_names) - 1:
            plt.xlabel('Time (s)')
        plt.legend(loc='upper right')
    plt.tight_layout()
    plt.savefig(os.path.join(data_dir, filename))
    plt.close()

# Data augmentation using sliding windows
def augment_data(data, labels, window_size, step_size):
    augmented_data = []
    augmented_labels = []
    for i in range(data.shape[0]):  # Iterate over each file
        eeg_data = data[i]  # Shape: (n_channels, n_samples)
        n_windows = int((eeg_data.shape[1] - window_size) / step_size) + 1
        for start in range(0, eeg_data.shape[1] - window_size + 1, int(step_size)):
            window = eeg_data[:, start:start + window_size]
            augmented_data.append(window)
            augmented_labels.append(labels[i])
    return np.array(augmented_data), np.array(augmented_labels)

# Create 1D-CNN model (Ternary Classification)
def create_1dcnn(input_shape):
    model = Sequential([
        Conv1D(128, kernel_size=5, activation='relu', padding='same', input_shape=input_shape),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(64, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Conv1D(32, kernel_size=5, activation='relu', padding='same'),
        BatchNormalization(),
        MaxPooling1D(pool_size=2),
        Dropout(0.3),
        Flatten(),
        Dense(128, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dropout(0.4),
        Dense(64, activation='relu', kernel_regularizer=tf.keras.regularizers.l2(0.01)),
        Dense(3, activation='softmax')  # Ternary classification (3 classes)
    ])
    model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=0.0001),
                  loss='categorical_crossentropy',
                  metrics=['accuracy'])
    return model

# Main pipeline
def main():
    logger.info("Starting EEG data processing...")

    # Step 1: Data Labeling
    try:
        labels_df = create_labels_file(data_dir)
    except Exception as e:
        logger.error(f"Failed to create labels file: {str(e)}")
        return

    # Load and preprocess data
    try:
        data, labels, subject_ids, conditions = load_eeg_data(data_dir, labels_df)
    except Exception as e:
        logger.error(f"Failed to load data: {str(e)}")
        return
    logger.info(f"Loaded {len(data)} valid files.")

    start_time = time.time()
    all_preprocessed_dfs = []

    # Preprocess data (only bandpass filtering)
    for i, (eeg_data, subject_id, condition) in enumerate(zip(data, subject_ids, conditions)):
        # Apply bandpass filter
        filtered_data = bandpass_filter(eeg_data, lowcut=1.0, highcut=40.0)
        
        # Plot channels for the first file
        if i == 0:
            plot_eeg_channels(filtered_data, ch_names, fs)
            logger.info("Plotted channels for first file.")

        # Save bandpass filtered data for this file
        df = pd.DataFrame(filtered_data.T, columns=ch_names)
        df['Subject_ID'] = subject_id
        df['Condition'] = condition
        df['Label'] = labels[i]
        df['Sample_Index'] = range(len(df))
        all_preprocessed_dfs.append(df)
        logger.info(f"Preprocessed file {i+1} (Subject: {subject_id}, Condition: {condition})")

    # Save all bandpass filtered data to extracted.csv
    if all_preprocessed_dfs:
        all_preprocessed_df = pd.concat(all_preprocessed_dfs, ignore_index=True)
        all_preprocessed_df.to_csv(os.path.join(data_dir, 'extracted.csv'), index=False)
        logger.info(f"Saved preprocessed data for {len(all_preprocessed_dfs)} files to extracted.csv (Shape: {all_preprocessed_df.shape})")
    else:
        logger.error("No preprocessed data to save to extracted.csv.")
        return

    # Data augmentation
    X, y = augment_data(data, labels, window_size, step_size)
    logger.info(f"After augmentation, dataset size: {X.shape[0]} samples (shape: {X.shape})")

    # Reshape X to (n_samples, timesteps, channels) for Conv1D
    X = X.transpose(0, 2, 1)  # From (n_samples, 14, 512) to (n_samples, 512, 14)
    logger.info(f"Reshaped X to: {X.shape}")

    # Convert labels to categorical (one-hot encoded) for ternary classification
    y_cat = tf.keras.utils.to_categorical(y, num_classes=3)

    # Standardize the data
    scaler = StandardScaler()
    X_reshaped = X.reshape(X.shape[0], -1)  # Flatten for scaling
    X_scaled = scaler.fit_transform(X_reshaped)
    X = X_scaled.reshape(X.shape)  # Reshape back to (n_samples, timesteps, channels)

    # 5-fold cross-validation with 1D-CNN
    skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=123)
    fold = 1
    val_losses = []
    val_accuracies = []

    for train_idx, val_idx in skf.split(X, y):
        logger.info(f"Training fold {fold}...")
        X_train, X_val = X[train_idx], X[val_idx]
        y_train, y_val = y_cat[train_idx], y_cat[val_idx]

        # Create and train 1D-CNN model
        model = create_1dcnn(input_shape=(window_size, n_channels))  # (timesteps, channels) = (512, 14)
        
        history = model.fit(
            X_train, y_train,
            epochs=250,
            batch_size=32,
            validation_data=(X_val, y_val),
            verbose=1
        )

        # Collect validation metrics
        val_losses.append(history.history['val_loss'])
        val_accuracies.append(history.history['val_accuracy'])

        # Evaluate on validation set
        val_pred = model.predict(X_val, verbose=0)
        val_pred_classes = np.argmax(val_pred, axis=1)
        val_true_classes = np.argmax(y_val, axis=1)
        val_accuracy = accuracy_score(val_true_classes, val_pred_classes)
        logger.info(f"Fold {fold} Validation Accuracy: {val_accuracy:.4f}")

        # Plot confusion matrix for this fold with ternary labels
        plt.figure(figsize=(5, 4))
        cm = confusion_matrix(val_true_classes, val_pred_classes, labels=[0, 1, 2])
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=['Low (0)', 'Medium (1)', 'High (2)'], yticklabels=['Low (0)', 'Medium (1)', 'High (2)'])
        plt.title(f'Confusion Matrix - Fold {fold}')
        plt.xlabel('Predicted')
        plt.ylabel('True')
        plt.savefig(os.path.join(data_dir, f'confusion_matrix_fold_{fold}.png'))
        plt.close()

        fold += 1

    # Plot validation loss across folds
    plt.figure(figsize=(10, 5))
    for i, losses in enumerate(val_losses, 1):
        plt.plot(losses, label=f'Fold {i}')
    plt.title('Validation Loss Across Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Loss')
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(data_dir, 'validation_loss_ml.png'))
    plt.close()

    # Plot validation accuracy across folds
    plt.figure(figsize=(10, 5))
    for i, accuracies in enumerate(val_accuracies, 1):
        plt.plot(accuracies, label=f'Fold {i}')
    plt.title('Validation Accuracy Across Folds')
    plt.xlabel('Epoch')
    plt.ylabel('Accuracy')
    plt.grid(True)
    plt.legend()
    plt.savefig(os.path.join(data_dir, 'validation_accuracy_ml.png'))
    plt.close()

    # Compute average validation loss and accuracy across folds
    # Align lengths by finding the shortest history and trimming/padding others
    min_length = min(len(losses) for losses in val_losses)
    aligned_val_losses = []
    aligned_val_accuracies = []
    
    for losses, accuracies in zip(val_losses, val_accuracies):
        # Trim or pad losses
        if len(losses) > min_length:
            aligned_val_losses.append(losses[:min_length])
        else:
            # Pad with the last value
            padded_losses = losses + [losses[-1]] * (min_length - len(losses))
            aligned_val_losses.append(padded_losses)
        
        # Trim or pad accuracies
        if len(accuracies) > min_length:
            aligned_val_accuracies.append(accuracies[:min_length])
        else:
            # Pad with the last value
            padded_accuracies = accuracies + [accuracies[-1]] * (min_length - len(accuracies))
            aligned_val_accuracies.append(padded_accuracies)

    # Convert to numpy arrays and compute averages
    aligned_val_losses = np.array(aligned_val_losses)
    aligned_val_accuracies = np.array(aligned_val_accuracies)
    avg_val_loss = np.mean(aligned_val_losses, axis=0)
    avg_val_accuracy = np.mean(aligned_val_accuracies, axis=0)

    # Plot average validation loss and accuracy on the same graph
    fig, ax1 = plt.subplots(figsize=(10, 5))
    
    # Plot average validation accuracy on the left y-axis
    color_acc = 'tab:blue'
    ax1.set_xlabel('Epoch')
    ax1.set_ylabel('Average Validation Accuracy', color=color_acc)
    ax1.plot(range(1, len(avg_val_accuracy) + 1), avg_val_accuracy, label='Avg Validation Accuracy', color=color_acc)
    ax1.tick_params(axis='y', labelcolor=color_acc)
    ax1.grid(True)

    # Create a second y-axis for average validation loss
    ax2 = ax1.twinx()
    color_loss = 'tab:orange'
    ax2.set_ylabel('Average Validation Loss', color=color_loss)
    ax2.plot(range(1, len(avg_val_loss) + 1), avg_val_loss, label='Avg Validation Loss', color=color_loss)
    ax2.tick_params(axis='y', labelcolor=color_loss)

    # Add title and legend
    plt.title('Average Validation Loss and Accuracy Across Folds')
    fig.legend(loc='upper right', bbox_to_anchor=(0.95, 0.95))
    plt.savefig(os.path.join(data_dir, 'avg_val_loss_accuracy.png'))
    plt.close()
    logger.info("Saved average validation loss and accuracy plot: avg_val_loss_accuracy.png")

    logger.info(f"Total processing time: {time.time() - start_time:.2f} seconds")

if __name__ == "__main__":
    main()

2025-06-05 01:29:57,556 - INFO - Starting EEG data processing...
2025-06-05 01:29:57,559 - INFO - Subject 01: class_lo = 0, class_hi = 2
2025-06-05 01:29:57,560 - INFO - Subject 02: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,561 - INFO - Subject 03: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,562 - INFO - Subject 04: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,563 - INFO - Subject 06: class_lo = 1, class_hi = 2
2025-06-05 01:29:57,564 - INFO - Subject 07: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,565 - INFO - Subject 08: class_lo = 0, class_hi = 2
2025-06-05 01:29:57,566 - INFO - Subject 09: class_lo = 0, class_hi = 2
2025-06-05 01:29:57,566 - INFO - Subject 10: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,567 - INFO - Subject 11: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,569 - INFO - Subject 12: class_lo = 0, class_hi = 1
2025-06-05 01:29:57,570 - INFO - Subject 13: class_lo = 0, class_hi = 2
2025-06-05 01:29:57,571 - INFO - Subject 14: class_lo = 0, class_hi = 2

Epoch 1/250
167/167 [==============================] - 18s 88ms/step - loss: 4.6067 - accuracy: 0.4093 - val_loss: 4.3267 - val_accuracy: 0.3551
Epoch 2/250
167/167 [==============================] - 16s 95ms/step - loss: 4.2336 - accuracy: 0.4533 - val_loss: 4.2917 - val_accuracy: 0.3941
Epoch 3/250
167/167 [==============================] - 14s 82ms/step - loss: 3.9885 - accuracy: 0.5180 - val_loss: 4.1458 - val_accuracy: 0.4730
Epoch 4/250
167/167 [==============================] - 14s 82ms/step - loss: 3.7686 - accuracy: 0.5606 - val_loss: 3.9741 - val_accuracy: 0.5135
Epoch 5/250
167/167 [==============================] - 14s 82ms/step - loss: 3.5852 - accuracy: 0.5811 - val_loss: 3.8053 - val_accuracy: 0.5315
Epoch 6/250
167/167 [==============================] - 13s 80ms/step - loss: 3.3749 - accuracy: 0.6273 - val_loss: 3.5950 - val_accuracy: 0.5788
Epoch 7/250
167/167 [==============================] - 14s 83ms/step - loss: 3.2056 - accuracy: 0.6351 - val_loss: 3.4191 - val_ac

2025-06-05 02:28:56,010 - INFO - Fold 1 Validation Accuracy: 0.9039
2025-06-05 02:28:56,487 - INFO - Training fold 2...


Epoch 1/250
167/167 [==============================] - 18s 84ms/step - loss: 4.4636 - accuracy: 0.4058 - val_loss: 4.2963 - val_accuracy: 0.2973
Epoch 2/250
167/167 [==============================] - 14s 82ms/step - loss: 4.1412 - accuracy: 0.4546 - val_loss: 3.9609 - val_accuracy: 0.5210
Epoch 3/250
167/167 [==============================] - 13s 81ms/step - loss: 3.9028 - accuracy: 0.5056 - val_loss: 3.6635 - val_accuracy: 0.6179
Epoch 4/250
167/167 [==============================] - 13s 81ms/step - loss: 3.6583 - accuracy: 0.5597 - val_loss: 3.5113 - val_accuracy: 0.5601
Epoch 5/250
167/167 [==============================] - 13s 81ms/step - loss: 3.4361 - accuracy: 0.5931 - val_loss: 3.2888 - val_accuracy: 0.6246
Epoch 6/250
167/167 [==============================] - 14s 81ms/step - loss: 3.2402 - accuracy: 0.6248 - val_loss: 3.2236 - val_accuracy: 0.6044
Epoch 7/250
167/167 [==============================] - 13s 81ms/step - loss: 3.0504 - accuracy: 0.6421 - val_loss: 3.0759 - val_ac

2025-06-05 03:24:35,004 - INFO - Fold 2 Validation Accuracy: 0.8934
2025-06-05 03:24:35,242 - INFO - Training fold 3...


...

2025-06-05 04:20:01,685 - INFO - Fold 3 Validation Accuracy: 0.8956
2025-06-05 04:20:01,903 - INFO - Training fold 4...


Epoch 1/250
167/167 [==============================] - 18s 86ms/step - loss: 4.5908 - accuracy: 0.4047 - val_loss: 4.4255 - val_accuracy: 0.2920
Epoch 2/250
167/167 [==============================] - 14s 82ms/step - loss: 4.2637 - accuracy: 0.4362 - val_loss: 4.2078 - val_accuracy: 0.3716
Epoch 3/250
167/167 [==============================] - 14s 82ms/step - loss: 4.0310 - accuracy: 0.4910 - val_loss: 4.1742 - val_accuracy: 0.4114
Epoch 4/250
167/167 [==============================] - 13s 80ms/step - loss: 3.8122 - accuracy: 0.5336 - val_loss: 4.0078 - val_accuracy: 0.4264
Epoch 5/250
167/167 [==============================] - 14s 82ms/step - loss: 3.5942 - accuracy: 0.5745 - val_loss: 3.9164 - val_accuracy: 0.4467
Epoch 6/250
167/167 [==============================] - 14s 83ms/step - loss: 3.4216 - accuracy: 0.5983 - val_loss: 3.8831 - val_accuracy: 0.4715
Epoch 7/250
167/167 [==============================] - 14s 81ms/step - loss: 3.2389 - accuracy: 0.6246 - val_loss: 3.5581 - val_ac

2025-06-05 07:34:15,212 - INFO - Fold 4 Validation Accuracy: 0.8956
2025-06-05 07:34:15,624 - INFO - Training fold 5...


Epoch 1/250
167/167 [==============================] - 18s 85ms/step - loss: 4.6349 - accuracy: 0.3913 - val_loss: 4.3879 - val_accuracy: 0.3063
Epoch 2/250
167/167 [==============================] - 13s 79ms/step - loss: 4.2465 - accuracy: 0.4369 - val_loss: 4.1814 - val_accuracy: 0.3799
Epoch 3/250
167/167 [==============================] - 13s 81ms/step - loss: 3.9966 - accuracy: 0.4976 - val_loss: 4.0449 - val_accuracy: 0.4369
Epoch 4/250
167/167 [==============================] - 14s 87ms/step - loss: 3.7873 - accuracy: 0.5289 - val_loss: 3.8586 - val_accuracy: 0.5135
Epoch 5/250
167/167 [==============================] - 14s 86ms/step - loss: 3.5655 - accuracy: 0.5745 - val_loss: 3.5783 - val_accuracy: 0.5773
Epoch 6/250
167/167 [==============================] - 14s 86ms/step - loss: 3.3691 - accuracy: 0.6015 - val_loss: 3.4519 - val_accuracy: 0.5773
Epoch 7/250
167/167 [==============================] - 14s 85ms/step - loss: 3.1750 - accuracy: 0.6325 - val_loss: 3.2508 - val_ac

2025-06-05 08:32:57,353 - INFO - Fold 5 Validation Accuracy: 0.8979
2025-06-05 08:32:58,751 - INFO - Saved average validation loss and accuracy plot: avg_val_loss_accuracy.png
2025-06-05 08:32:58,752 - INFO - Total processing time: 25319.40 seconds
